# Vanilla LLM + Critic Experiment

Runs a vanilla LLM along with a critic to generate answers for medical questions
This experiment is done to check the improvement without a RAG and just with a critic agent

**Pipeline:** Question -> Generator -> Critic -> Revision → Answer

**Evaluation:** RAGAS and DeepEval metrics

In [5]:
import sys
sys.path.append("..")

import os
import time
import json
import pandas as pd
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from datasets import load_dataset
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
import config
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from ast import literal_eval
from deepeval.evaluate import DisplayConfig, AsyncConfig
from ragas.metrics import NonLLMContextRecall, NonLLMContextPrecisionWithReference, BleuScore, RougeScore
from ragas import SingleTurnSample
from deepeval.test_case import LLMTestCase
from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import (
    FaithfulnessMetric, ContextualRecallMetric,
    ContextualPrecisionMetric, AnswerRelevancyMetric,
)
import deepeval
import instructor
from groq import AsyncGroq

from ragas.llms.base import InstructorLLM
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams
os.environ["CONFIDENT_TRACE_VERBOSE"] = "0"
os.environ["DEEPEVAL_VERBOSE_MODE"] = "0"
os.environ["DEEPEVAL_RETRY_MAX_ATTEMPTS"] = "2"

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_23165/1774783705.py:17: DeprecationWarning: Importing NonLLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import NonLLMContextRecall
  from ragas.metrics import NonLLMContextRecall, NonLLMContextPrecisionWithReference, BleuScore, RougeScore
/tmp/ipykernel_23165/1774783705.py:17: DeprecationWarning: Importing NonLLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metric

## LLM Object and GroqKeyRotator

In [6]:
def get_groq_llm(model=None, api_key=None):
    return ChatGroq(
        model=model or config.LLM_MODEL,
        api_key=api_key or config.GROQ_API_KEY,
    )


class GroqKeyRotator:
    """Cycles through config.GROQ_API_KEYS, rebuilding the LLM client on each rotation."""

    def __init__(self, model=None):
        if not config.GROQ_API_KEYS:
            raise ValueError("config.GROQ_API_KEYS is empty — set GROQ_API_KEY or GROQ_API_KEYS in .env")
        self.api_keys = config.GROQ_API_KEYS
        self.model = model or config.LLM_MODEL
        self.current_idx = 0
        print(f"Initialized GroqKeyRotator with {len(self.api_keys)} API key(s)")

    def get_llm(self):
        """Return a ChatGroq instance using the current API key."""
        return ChatGroq(
            model=self.model,
            api_key=self.api_keys[self.current_idx],
        )

    def rotate(self):
        """Advance to the next key, wrapping around."""
        self.current_idx = (self.current_idx + 1) % len(self.api_keys)
        print(f"Rotated to API key index {self.current_idx}")

## LLM Tempates and Building the LLM Agent

In [18]:
VANILLA_PROMPT_TEMPLATE = """Answer the following biomedical research question using your existing knowledge.
Question: {question}

Provide a precise, evidence-based answer. Do not speculate beyond established scientific knowledge.
Answer:"""

VANILLA_PROMPT = PromptTemplate(
    template=VANILLA_PROMPT_TEMPLATE,
    input_variables=["question"],
)

CRITIC_PROMPT_TEMPLATE = """You are a biomedical expert reviewer trained in evaluation biomedical Question Answering tasks.
Question: {question}

Answer: {answer}

Review the answer for scientific accuracy, completeness, unsupported claims, ambiguity, or missing evidence.
If no major issues exist, say "No major issues." Return proper critique in 2 to 3 sentences only.

Critique:"""

CRITIC_PROMPT = PromptTemplate(
    input_variables=["question", "answer"],
    template=CRITIC_PROMPT_TEMPLATE,
)

REVISION_PROMPT_TEMPLATE = """Revise the answer using the critique.

Question:
{question}

Original Answer:
{answer}

Critique:
{critique}

Provide an improved, precise, evidence-based final answer.

Final Answer:"""

REVISION_PROMPT = PromptTemplate(
    input_variables=["question", "answer", "critique"],
    template=REVISION_PROMPT_TEMPLATE,
)

def build_llm_critic_agent(llm):
    """Method to build a generator-critic-refinement agent chain.

    Constructs a three-stage LangChain pipeline where an initial answer is first
    generated using the base LLM, then reviewed by a critic agent for
    biomedical quality issues, and finally revised using the generated critique.

    Pipeline:
        Question -> Generator -> Critic -> Revision

    Args:
        llm: LangChain chat model instance used for generation,
            critique, and refinement.

    Returns:
        LangChain runnable agent chain.
    """
    generator_chain = VANILLA_PROMPT | llm
    critic_chain = CRITIC_PROMPT | llm
    revision_chain = REVISION_PROMPT | llm
    return (
        {"question": RunnablePassthrough()}
        | RunnableLambda(
            lambda x: {
                **x,
                "generator_result": generator_chain.invoke({"question": x["question"]}),
            }
        )
        | RunnableLambda(
            lambda x: {
                **x,
                "answer": x["generator_result"].content,
                "critic_result": critic_chain.invoke({
                    "question": x["question"],
                    "answer": x["generator_result"].content,
                }),
            }
        )
        | RunnableLambda(
            lambda x: {
                **x,
                "critique": x["critic_result"].content,
                "final_result": revision_chain.invoke({
                    "question": x["question"],
                    "answer": x["answer"],
                    "critique": x["critic_result"].content,
                }),
            }
        )
    )


def get_tokens(result):
    """Helper Method which returns a tuple of (prompt_tokens, completion_tokens)."""
    usage = getattr(result, "usage_metadata", None) or result.response_metadata.get("token_usage", {})
    prompt = usage.get("prompt_tokens") or usage.get("input_tokens") or 0
    completion = usage.get("completion_tokens") or usage.get("output_tokens") or 0
    return prompt, completion

def _run_slice(slice_df, api_key, model, delay, key_idx):
    """Process a contiguous slice of the evaluation set using a single dedicated API key.

    Called in its own thread by run_rag_parallel. Rows are processed sequentially
    with `delay` seconds between requests to stay within the key's daily quota.
    Adds generated_answer as new column to a copy of slice_df.

    Args:
        slice_df: Contiguous DataFrame slice assigned to this key.
        api_key: Groq API key string dedicated to this thread.
        model: LLM model name forwarded from GroqKeyRotator.
        delay: Seconds to sleep between rows.
        key_idx: Key index used only for log prefixes.

    Returns:
        Copy of slice_df with one new columns: generated_answer.
    """
    chain = build_llm_critic_agent(ChatGroq(model=model, api_key=api_key))
    result_df = slice_df.copy().reset_index(drop=True)
    initial_answers = [None] * len(slice_df)
    critiques = [None] * len(slice_df)
    total_time_list = [None] * len(slice_df)
    prompt_tokens_list = [None] * len(slice_df)
    completion_tokens_list = [None] * len(slice_df)
    total_tokens_list = [None] * len(slice_df)
    # Final Generated Answers
    generated_answer_list = [None] * len(slice_df)

    for row_idx, (_, row) in enumerate(slice_df.iterrows()):
        question = row["question"]
        try:
            time_start = time.perf_counter()
            result = chain.invoke(question)
            time_end = time.perf_counter()
            total_time_list[row_idx] = time_end - time_start

            # Calculating Token Usage
            p1, c1 = get_tokens(result["generator_result"])
            p2, c2 = get_tokens(result["critic_result"])
            p3, c3 = get_tokens(result["final_result"])
            prompt_tokens = p1 + p2 + p3
            completion_tokens = c1 + c2 + c3

            initial_answers[row_idx] = result["answer"]
            critiques[row_idx] = result["critique"]
            generated_answer_list[row_idx] = result["final_result"].content
            total_tokens_list[row_idx] = prompt_tokens + completion_tokens
            prompt_tokens_list[row_idx] = prompt_tokens
            completion_tokens_list[row_idx] = completion_tokens
        except Exception as e:
            print(f"[Key {key_idx}] Error on '{question[:50]}...': {e}")

        if row_idx < len(slice_df) - 1:
            time.sleep(delay)

    result_df["generated_answer"] = generated_answer_list
    result_df["initial_answer"] = initial_answers
    result_df["critique"] = critiques
    result_df["total_time"] = total_time_list
    result_df["prompt_tokens"] = prompt_tokens_list
    result_df["completion_tokens"] = completion_tokens_list
    result_df["total_tokens"] = total_tokens_list

    completed = sum(1 for x in generated_answer_list if x is not None)
    print(f"[Key {key_idx}] Done — {completed}/{len(slice_df)} rows collected")
    return result_df


def run_llm_parallel(df, key_rotator, rows_per_key=None, delay=None):
    """Assign a contiguous slice of rows to each API key and run all slices in parallel.

    Key 0 gets rows 0..rows_per_key-1, key 1 gets the next rows_per_key rows, etc.
    Each key runs in its own thread; since Groq keys have independent daily quotas,
    the threads don't interfere with each other.

    ThreadPoolExecutor is used (not asyncio) because: (1) network I/O releases the GIL
    so threads genuinely run concurrently, (2) Jupyter/Colab already have a running event
    loop so asyncio.run() raises RuntimeError, and (3) ChatGroq.invoke() is synchronous.

    Args:
        df: DataFrame with at least question and golden_answer columns.
        key_rotator: GroqKeyRotator providing API keys and LLM model name.
        rows_per_key: Max rows assigned to each key (default config.PARALLEL_BUCKET_SIZE).
        delay: Seconds between rows within each key's slice (default config.PARALLEL_DELAY_SECONDS).

    Returns:
        a copy of df with one new column - generated_answer - in original row order.
    """
    rows_per_key = rows_per_key or config.PARALLEL_BUCKET_SIZE
    delay = delay or config.PARALLEL_DELAY_SECONDS
    api_keys = key_rotator.api_keys

    if len(df) == 0:
        raise ValueError("DataFrame is empty — nothing to evaluate.")

    total_capacity = len(api_keys) * rows_per_key
    if len(df) > total_capacity:
        print(
            f"Warning: {len(df)} rows exceed capacity ({len(api_keys)} keys × {rows_per_key} rows = "
            f"{total_capacity}). Only the first {total_capacity} rows will be processed."
        )
        df = df.iloc[:total_capacity]

    slices = []
    for i, key in enumerate(api_keys):
        start = i * rows_per_key
        if start >= len(df):
            break
        slices.append((key, i, df.iloc[start: start + rows_per_key]))

    print(f"\n{len(df)} rows split across {len(slices)} key(s) ({rows_per_key} rows/key max):")
    for key, i, s in slices:
        print(f"  Key {i}: rows {i * rows_per_key}–{i * rows_per_key + len(s) - 1} ({len(s)} rows)")
    print()

    ordered_results = [None] * len(slices)

    with ThreadPoolExecutor(max_workers=len(slices)) as executor:
        future_to_idx = {
            executor.submit(
                _run_slice,
                s, key, key_rotator.model, delay, i,
            ): idx
            for idx, (key, i, s) in enumerate(slices)
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            _, _, s = slices[idx]
            try:
                ordered_results[idx] = future.result()
            except Exception as e:
                print(f"[Key {idx}] Thread failed: {e}")
                fallback = s.copy().reset_index(drop=True)
                fallback["generated_answer"] = [None] * len(s)
                ordered_results[idx] = fallback

    final_df = pd.concat(ordered_results, ignore_index=True)
    completed = final_df["generated_answer"].notna().sum()
    print(f"\nCompleted {completed}/{len(df)} questions total")
    print(f"\nAverage Time Per Query : {sum(final_df["total_time"])/len(final_df)}")
    print(f"\nAverage Total Tokens Per Query : {sum(final_df["total_tokens"])/len(final_df)}")
    return final_df

## Evaluation Functions

In [33]:
class GroqModel(DeepEvalBaseLLM):
    def __init__(self, model=None):
        self.model = model or get_groq_llm()

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        response = self.model.invoke(prompt)
        return response.content

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return "Groq Model"


def build_test_cases(eval_df):
    return [
        LLMTestCase(
            input=row["question"],
            actual_output=row["generated_answer"],
            expected_output=row["golden_answer"],
        )
        for _, row in eval_df.iterrows()
    ]


def _make_ragas_scores_df(all_scores, metric_name):
    """Return a minimal DataFrame with question_index and metric score."""
    return pd.DataFrame({
        "question_index": range(len(all_scores)),
        metric_name: all_scores,
    })


def evaluate_ragas(eval_df, metric, results_file=None):
    """Evaluate with a non-LLM RAGAS metric over each row of the eval DataFrame.

    Uses SingleTurnSample + single_turn_score (synchronous) — no API keys required.
    Compares golden answer and generated answer.

    Args:
        eval_df: DataFrame produced by run_rag / run_rag_parallel.
            Expected columns: question, generated_answer,
            golden_contexts, golden_answer.
        metric: A RAGAS metric instance.
        results_file: Optional CSV path to save per-sample scores (question_index + score only).

    Returns:
        Tuple of (list of per-sample scores, average score, scores DataFrame).
    """
    metric_name = type(metric).__name__
    all_scores = []

    for _, row in eval_df.iterrows():
        sample = SingleTurnSample(
            user_input=row["question"],
            reference_contexts=row["golden_contexts"],
            reference=row["golden_answer"],
            response=row["generated_answer"]
        )
        score = metric.single_turn_score(sample)
        all_scores.append(score)

    avg = sum(all_scores) / len(all_scores)
    print(f"\n=== {metric_name}: {avg:.4f} (avg over {len(all_scores)} samples) ===")

    scores_df = _make_ragas_scores_df(all_scores, metric_name)

    if results_file:
        scores_df.to_csv(results_file, index=False)
        print(f"Saved scores to {results_file}")

    return all_scores, avg, scores_df


def build_ragas_combined(eval_df, score_dfs, results_file=None):
    """Combine eval_df with per-metric score DataFrames into one summary CSV.

    Args:
        eval_df: DataFrame produced by run_rag / run_rag_parallel.
        score_dfs: List of score DataFrames from evaluate_ragas, each with
            question_index + one metric score column.
        results_file: Optional CSV path to save the combined DataFrame.

    Returns:
        Combined DataFrame with question_index, question,
        golden_contexts, golden_answer, generated_response, and one column per metric.
    """
    combined = eval_df.copy().reset_index(drop=True)
    combined.insert(0, "question_index", range(len(combined)))
    combined = combined.rename(columns={"generated_answer": "generated_response"})

    for scores_df in score_dfs:
        combined = combined.merge(scores_df, on="question_index", how="left")

    if results_file:
        combined.to_csv(results_file, index=False)
        print(f"Saved combined RAGAS results to {results_file}")

    return combined


def _run_deepeval_slice(test_case_slice, api_key, model, metric_cls, threshold, delay, key_idx, metric_kwargs=None):
    """Evaluate a contiguous slice of test cases using a single dedicated API key.

    Called in its own thread by evaluate_deepeval_parallel. Test cases are evaluated
    sequentially with `delay` seconds between each to stay within the key's daily quota.

    Args:
        test_case_slice: List of LLMTestCase objects assigned to this key.
        api_key: Groq API key string dedicated to this thread.
        model: LLM model name forwarded from GroqKeyRotator.
        metric_cls: DeepEval metric class (e.g. ContextualRecallMetric), not an instance.
        threshold: Pass/fail threshold for the metric.
        delay: Seconds to sleep between test cases.
        key_idx: Key index used only for log prefixes.

    Returns:
        List of deepeval test results in slice order.
    """
    llm = ChatGroq(model=model, api_key=api_key)
    results = []
    metric_kwargs = {} if metric_kwargs is None else dict(metric_kwargs)

    for i, test_case in enumerate(test_case_slice):
        try:
            metric = metric_cls(threshold=threshold, model=GroqModel(model=llm),
                                **metric_kwargs)
            result = deepeval.evaluate([test_case], metrics=[metric],
                                       display_config= DisplayConfig(
                                           verbose_mode=False,
                                           show_indicator=False,
                                           print_results=False),
                                       async_config=AsyncConfig(
                                           run_async=False
                                       ))
            results.extend(result.test_results)
        except Exception as e:
            print(f"[Key {key_idx}] Error on case {i + 1}/{len(test_case_slice)}: {e}")

        if i < len(test_case_slice) - 1:
            time.sleep(delay)

    print(f"[Key {key_idx}] Done — {len(results)}/{len(test_case_slice)} cases evaluated")
    return results


def evaluate_deepeval_parallel(test_cases, metric_cls, key_rotator, threshold=0.5, results_file=None, delay=None, rows_per_key=None, metric_kwargs=None):
    """Assign a contiguous slice of test cases to each API key and run all slices in parallel.

    Key 0 gets test_cases[0:rows_per_key], key 1 gets the next slice, etc.
    Each key runs in its own thread; since Groq keys have independent daily quotas,
    the threads don't interfere with each other.

    Args:
        test_cases: List of LLMTestCase objects built by build_test_cases.
        metric_cls: DeepEval metric class (e.g. ContextualRecallMetric), not an instance.
        key_rotator: GroqKeyRotator providing API keys and LLM model name.
        threshold: Pass/fail threshold for the metric (default 0.5).
        results_file: Optional CSV path to save per-sample scores.
        delay: Seconds between cases within each slice (defaults to config.DEEPEVAL_DELAY_SECONDS).
        rows_per_key: Max cases assigned to each key (default config.PARALLEL_BUCKET_SIZE).

    Returns:
        List of deepeval test results in original case order.
    """
    rows_per_key = rows_per_key or config.PARALLEL_BUCKET_SIZE
    delay = delay or config.DEEPEVAL_DELAY_SECONDS
    api_keys = key_rotator.api_keys
    metric_kwargs = {} if metric_kwargs is None else dict(metric_kwargs)

    if not test_cases:
        raise ValueError("test_cases is empty — nothing to evaluate.")

    total_capacity = len(api_keys) * rows_per_key
    if len(test_cases) > total_capacity:
        print(
            f"Warning: {len(test_cases)} cases exceed capacity ({len(api_keys)} keys × {rows_per_key} = "
            f"{total_capacity}). Only the first {total_capacity} cases will be processed."
        )
        test_cases = test_cases[:total_capacity]

    slices = []
    for i, key in enumerate(api_keys):
        start = i * rows_per_key
        if start >= len(test_cases):
            break
        slices.append((key, i, test_cases[start: start + rows_per_key]))

    print(f"\n{len(test_cases)} cases split across {len(slices)} key(s) ({rows_per_key} cases/key max):")
    for key, i, s in slices:
        print(f"  Key {i}: cases {i * rows_per_key}–{i * rows_per_key + len(s) - 1} ({len(s)} cases)")
    print()

    ordered_results = [None] * len(slices)

    with ThreadPoolExecutor(max_workers=len(slices)) as executor:
        future_to_idx = {
            executor.submit(
                _run_deepeval_slice,
                s, key, key_rotator.model,
                metric_cls, threshold, delay, i, metric_kwargs
            ): idx
            for idx, (key, i, s) in enumerate(slices)
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            try:
                ordered_results[idx] = future.result()
            except Exception as e:
                print(f"[Key {idx}] Thread failed: {e}")
                ordered_results[idx] = []

    all_results = []
    for key_results in ordered_results:
        all_results.extend(key_results)

    if all_results:
        scores = [r.metrics_data[0].score for r in all_results]
        metric_name = all_results[0].metrics_data[0].name
        average = sum(scores) / len(scores)
        print(f"\n=== {metric_name}: {average:.4f} (avg over {len(scores)} samples) ===")

        rows = []
        for r in all_results:
            rows.append({
                "question": r.input,
                "generated_answer": r.actual_output,
                "retrieved_contexts": r.retrieval_context,
                "golden_answer": r.expected_output,
                r.metrics_data[0].name: r.metrics_data[0].score,
            })
        res_df = pd.DataFrame(rows)
        if results_file:
            res_df.to_csv(results_file, index=False)
            print(f"Saved to {results_file}")

    return all_results, res_df


def resume_deepeval_from_csv(eval_dataset, existing_results_file, metric_cls, key_rotator,
    metric_column, threshold=0.5, delay=None, rows_per_key=None, metric_kwargs=None):
    existing_df = pd.read_csv(existing_results_file)

    completed_questions = set(existing_df.loc[
        existing_df[metric_column].notna(), "question"].astype(str))

    missing_df = eval_dataset[~eval_dataset["question"].astype(str)
                              .isin(completed_questions)].copy()
    if missing_df.empty:
        print("All rows already completed.")
        return existing_df
    print(f"Need to recompute {len(missing_df)} rows.")

    test_cases = build_test_cases(missing_df)

    _, new_results_df = evaluate_deepeval_parallel(test_cases, metric_cls,
        key_rotator, threshold=threshold, delay=delay, rows_per_key=rows_per_key,
        metric_kwargs=metric_kwargs)

    existing_df = existing_df[~existing_df["question"].astype(str).isin(
            new_results_df["question"].astype(str))]

    final_df = pd.concat([existing_df, new_results_df], ignore_index=True)
    if "question_idx" in final_df.columns:
        final_df = final_df.drop(columns=["question_idx"])
    final_df = final_df.merge(eval_dataset[["question", "question_idx"]], on="question", how="left")
    cols = ["question_idx"] + [c for c in final_df.columns if c != "question_idx"]
    final_df = final_df.sort_values("question_idx").reset_index(drop=True)

    final_df.to_csv(existing_results_file, index=False)
    print(f"Completed: {len(final_df)}/{len(eval_dataset)} rows")

    return final_df

---
## Setup

In [ ]:
key_rotator = GroqKeyRotator()
llm = key_rotator.get_llm()
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"LLM: {key_rotator.model}")
print(f"Run timestamp: {timestamp}")

Initialized GroqKeyRotator with 10 API key(s)
LLM: llama-3.3-70b-versatile
Run timestamp: 20260510_140037


## Prepare Evaluation Sample

Reads the pre-built golden dataset from `data/processed/golden_dataset_complete.csv`.
Generate this file once by running `python3 sampling.py` from the `research/` directory.
Keeping the split fixed is critical — regenerating mid-experiment would change which
questions each RAG variant sees, invalidating cross-experiment comparisons.

In [16]:
golden_df = pd.read_csv(config.DATA_PROCESSED_DIR / "golden_dataset_complete.csv")
print(f"Loaded {len(golden_df)} samples from golden_dataset_complete.csv")
print(f"Structure of golden dataset")
golden_df['golden_contexts'] = golden_df['golden_contexts'].apply(literal_eval)
print(type(golden_df['golden_contexts'].iloc[0]))
golden_df.info()

Loaded 200 samples from golden_dataset_complete.csv
Structure of golden dataset
<class 'list'>
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   question_idx     200 non-null    int64 
 1   question         200 non-null    object
 2   golden_answer    200 non-null    object
 3   golden_contexts  200 non-null    object
 4   query_type       200 non-null    object
 5   pubids_needed    200 non-null    object
dtypes: int64(1), object(5)
memory usage: 9.5+ KB


## Run Vanilla LLM + Critic Evaluation

In [ ]:
eval_dataset = run_llm_parallel(golden_df.head(1), key_rotator)
eval_dataset


1 rows split across 1 key(s) (20 rows/key max):
  Key 0: rows 0–0 (1 rows)

[Key 0] Done — 1/1 rows collected

Completed 1/1 questions total

Average Time Per Query : 5.2285315200001605

Average Total Tokens Per Query : 2734.0


,question_idx,question,golden_answer,golden_contexts,query_type,pubids_needed,generated_answer,initial_answer,critique,total_time,prompt_tokens,completion_tokens,total_tokens
0,0,Is there a relationship between rheumatoid art...,Based on data derived from self-reported healt...,"[1,412 individuals attending the University of...",Single-hop,['10783841'],"Yes, there is a relationship between rheumatoi...","Yes, there is a relationship between rheumatoi...",The answer accurately summarizes the current u...,5.228532,1190,1544,2734


In [ ]:
eval_dataset = run_llm_parallel(golden_df, key_rotator)
eval_dataset.to_csv(str(config.RESULTS_EVALSETS_DIR / f"vanilla_llm_critic_{timestamp}.csv"))
print(f"Generated {len(eval_dataset)} answers")


200 rows split across 10 key(s) (20 rows/key max):
  Key 0: rows 0–19 (20 rows)
  Key 1: rows 20–39 (20 rows)
  Key 2: rows 40–59 (20 rows)
  Key 3: rows 60–79 (20 rows)
  Key 4: rows 80–99 (20 rows)
  Key 5: rows 100–119 (20 rows)
  Key 6: rows 120–139 (20 rows)
  Key 7: rows 140–159 (20 rows)
  Key 8: rows 160–179 (20 rows)
  Key 9: rows 180–199 (20 rows)

[Key 9] Error on 'Do prior benign prostate biopsy and year of surger...': Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kr6pmr2ff3jbhg3emgs9qds4` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 10535, Requested 1712. Please try again in 1.234999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
[Key 6] Error on 'Does vaginal intraepithelial neoplasia have the sa...': Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama

In [23]:
key_rotator = GroqKeyRotator()

Initialized GroqKeyRotator with 10 API key(s)


In [14]:
error_rows = eval_dataset[eval_dataset['generated_answer'].isnull()]
print(f"Errors: {len(error_rows)}")
error_rows

Errors: 5


,Unnamed: 0,question_idx,question,golden_answer,golden_contexts,query_type,pubids_needed,generated_answer,initial_answer,critique,total_time,prompt_tokens,completion_tokens,total_tokens
95,95,95,Clinical identifiers for early-stage primary/i...,None of the clinical identifiers for early-sta...,[Adhesive capsulitis is often difficult to dia...,Single-hop,['24652474'],NaN,NaN,NaN,NaN,NaN,NaN,NaN
135,135,135,Does vaginal intraepithelial neoplasia have th...,These data suggest that CIN and VAIN may have ...,[Vaginal intraepithelial neoplasia is a little...,Single-hop,['18714572'],NaN,NaN,NaN,NaN,NaN,NaN,NaN
155,155,155,Do modifiable lifestyle and behavioral factors...,Modifiable lifestyle and behavioral factors we...,[In a five-year prospective study of 478 Black...,Causal,['22825590'],NaN,NaN,NaN,NaN,NaN,NaN,NaN
179,179,179,What were the AUC values of FDG-PET for detect...,"The AUC values were 0.967 (hyperglycaemic DM),...","[Total patients 219 (16 hyperglycaemic DM, 12 ...",Specificity,['15703931'],NaN,NaN,NaN,NaN,NaN,NaN,NaN
188,188,188,Do prior benign prostate biopsy and year of su...,Prior benign biopsy and year of surgery do not...,[Prior biopsy linked to more organ-confined di...,Comparative,"['15708048', '14713788']",NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
# Re-run for error rows if any
if len(error_rows) > 0:
    sub_golden_df = golden_df[golden_df['question_idx'].isin(error_rows['question_idx'])]
    sub_golden_df = sub_golden_df.reset_index(drop=True)
    sub_eval_dataset = run_llm_parallel(sub_golden_df, key_rotator, rows_per_key=1)

    for idx, row in sub_eval_dataset.iterrows():
        eval_dataset.at[row['question_idx'], 'generated_answer'] = row['generated_answer']
        eval_dataset.at[row['question_idx'], 'initial_answer'] = row['initial_answer']
        eval_dataset.at[row['question_idx'], 'critique'] = row['critique']
        eval_dataset.at[row['question_idx'], 'total_time'] = row['total_time']
        eval_dataset.at[row['question_idx'], 'prompt_tokens'] = row['prompt_tokens']
        eval_dataset.at[row['question_idx'], 'completion_tokens'] = row['completion_tokens']
        eval_dataset.at[row['question_idx'], 'total_tokens'] = row['total_tokens']

    eval_dataset.to_csv(str(config.RESULTS_EVALSETS_DIR / f"vanilla_llm_critic_{timestamp}.csv"))
    print(f"Remaining errors: {eval_dataset['generated_answer'].isnull().sum()}")
else:
    print("No errors to re-run.")


5 rows split across 5 key(s) (1 rows/key max):
  Key 0: rows 0–0 (1 rows)
  Key 1: rows 1–1 (1 rows)
  Key 2: rows 2–2 (1 rows)
  Key 3: rows 3–3 (1 rows)
  Key 4: rows 4–4 (1 rows)

[Key 4] Done — 1/1 rows collected
[Key 2] Done — 1/1 rows collected
[Key 3] Done — 1/1 rows collected
[Key 1] Done — 1/1 rows collected
[Key 0] Done — 1/1 rows collected

Completed 5/5 questions total

Average Time Per Query : 4.717694401399967

Average Total Tokens Per Query : 2554.2
Remaining errors: 0


In [25]:
print(f"Average Time Per Query : {sum(eval_dataset["total_time"])/len(eval_dataset)}")
print(f"Average Total Tokens Per Query : {sum(eval_dataset["total_tokens"])/len(eval_dataset)}")

Average Time Per Query : 7.238632663675028
Average Total Tokens Per Query : 2428.43


---
## RAGAS Evaluation

In [ ]:
ragas_blue_scores, ragas_blue_avg, ragas_blue_df = evaluate_ragas(
    eval_dataset, BleuScore(),
    results_file=str(config.RESULTS_RAGAS_DIR / f"vanilla_llm_critic_bleu_{timestamp}.csv")
)


=== BleuScore: 0.0849 (avg over 200 samples) ===
Saved scores to /content/results/ragas/vanilla_llm_critic_bleu_20260510_140037.csv


In [ ]:
ragas_rouge_scores, ragas_rouge_avg, ragas_rouge_df = evaluate_ragas(
    eval_dataset, RougeScore(rouge_type="rougeL", mode="fmeasure"),
    results_file=str(config.RESULTS_RAGAS_DIR / f"vanilla_llm_critic_rouge_{timestamp}.csv")
)


=== RougeScore: 0.0751 (avg over 200 samples) ===
Saved scores to /content/results/ragas/vanilla_llm_critic_rouge_20260510_140037.csv


In [ ]:
combined_ragas = build_ragas_combined(
    eval_dataset, [ragas_blue_df, ragas_rouge_df], results_file=str(config.RESULTS_RAGAS_DIR / f"vanilla_llm_critic_combined_{timestamp}.csv"))

Saved combined RAGAS results to /content/results/ragas/vanilla_llm_critic_combined_20260510_140037.csv


## DeepEval Evaluation

In [11]:
timestamp = "20260510_140037"
eval_dataset = pd.read_csv(str(config.RESULTS_EVALSETS_DIR / f"vanilla_llm_critic_{timestamp}.csv"))
eval_dataset['golden_contexts'] = eval_dataset['golden_contexts'].apply(literal_eval)
eval_dataset.head(2)

,Unnamed: 0,question_idx,question,golden_answer,golden_contexts,query_type,pubids_needed,generated_answer,initial_answer,critique,total_time,prompt_tokens,completion_tokens,total_tokens
0,0,0,Is there a relationship between rheumatoid art...,Based on data derived from self-reported healt...,"[1,412 individuals attending the University of...",Single-hop,['10783841'],"Yes, there is a significant relationship betwe...","Yes, there is a relationship between rheumatoi...","The answer is generally accurate, but it lacks...",4.773457,1073.0,1517.0,2590.0
1,1,1,"Do the changes in the serum levels of IL-2, IL...",The enhancement of serum TNFalpha and IL-6 lev...,[Acute pancreatitis is the major complication ...,Single-hop,['18670651'],"Yes, changes in serum levels of IL-2, IL-4, TN...","Yes, changes in serum levels of IL-2, IL-4, TN...",The answer accurately reflects the role of IL-...,3.714767,880.0,1195.0,2075.0


In [26]:
test_cases = build_test_cases(eval_dataset)

de_key_rotator = GroqKeyRotator(model="openai/gpt-oss-120b")

Initialized GroqKeyRotator with 10 API key(s)


In [32]:
deepeval_ar, de_ar_df = evaluate_deepeval_parallel(
    test_cases, AnswerRelevancyMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"vanilla_llm_critic_ans_relevancy_{timestamp}.csv")
)


200 cases split across 10 key(s) (20 cases/key max):
  Key 0: cases 0–19 (20 cases)
  Key 1: cases 20–39 (20 cases)
  Key 2: cases 40–59 (20 cases)
  Key 3: cases 60–79 (20 cases)
  Key 4: cases 80–99 (20 cases)
  Key 5: cases 100–119 (20 cases)
  Key 6: cases 120–139 (20 cases)
  Key 7: cases 140–159 (20 cases)
  Key 8: cases 160–179 (20 cases)
  Key 9: cases 180–199 (20 cases)



[Key 7] Done — 17/20 cases evaluated
[Key 8] Error on case 20/20: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01hzs3gv6vfsf9qg002zv18ps6` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198402, Requested 3144. Please try again in 11m7.872s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
[Key 8] Done — 14/20 cases evaluated

=== Answer Relevancy: 0.9466 (avg over 174 samples) ===
Saved to /content/results/deepeval/vanilla_llm_critic_ans_relevancy_20260510_140037.csv


In [34]:
answer_relevancy_df = resume_deepeval_from_csv(
    eval_dataset, str(config.RESULTS_DEEPEVAL_DIR / f"vanilla_llm_critic_ans_relevancy_{timestamp}.csv"),
    AnswerRelevancyMetric, de_key_rotator, "Answer Relevancy",
    rows_per_key=3
)

Need to recompute 26 rows.

26 cases split across 9 key(s) (3 cases/key max):
  Key 0: cases 0–2 (3 cases)
  Key 1: cases 3–5 (3 cases)
  Key 2: cases 6–8 (3 cases)
  Key 3: cases 9–11 (3 cases)
  Key 4: cases 12–14 (3 cases)
  Key 5: cases 15–17 (3 cases)
  Key 6: cases 18–20 (3 cases)
  Key 7: cases 21–23 (3 cases)
  Key 8: cases 24–25 (2 cases)

[Key 1] Error on case 1/3: 'NoneType' object has no attribute 'save'


[Key 6] Done — 3/3 cases evaluated

=== Answer Relevancy: 0.9908 (avg over 19 samples) ===
Completed: 193/200 rows


In [35]:
answer_relevancy_df = resume_deepeval_from_csv(
    eval_dataset, str(config.RESULTS_DEEPEVAL_DIR / f"vanilla_llm_critic_ans_relevancy_{timestamp}.csv"),
    AnswerRelevancyMetric, de_key_rotator, "Answer Relevancy",
    rows_per_key=2
)

Need to recompute 7 rows.

7 cases split across 4 key(s) (2 cases/key max):
  Key 0: cases 0–1 (2 cases)
  Key 1: cases 2–3 (2 cases)
  Key 2: cases 4–5 (2 cases)
  Key 3: cases 6–6 (1 cases)

[Key 0] Error on case 1/2: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kk3kjkb5f90vnh1k5kp1zmdm` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199716, Requested 1149. Please try again in 6m13.68s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


[Key 1] Done — 2/2 cases evaluated

=== Answer Relevancy: 0.7731 (avg over 5 samples) ===
Completed: 198/200 rows


In [40]:
de_key_rotator = GroqKeyRotator(model="openai/gpt-oss-120b")
answer_relevancy_df = resume_deepeval_from_csv(
    eval_dataset, str(config.RESULTS_DEEPEVAL_DIR / f"vanilla_llm_critic_ans_relevancy_{timestamp}.csv"),
    AnswerRelevancyMetric, de_key_rotator, "Answer Relevancy",
    rows_per_key=3
)

Initialized GroqKeyRotator with 7 API key(s)
Need to recompute 2 rows.

2 cases split across 1 key(s) (3 cases/key max):
  Key 0: cases 0–1 (2 cases)



[Key 0] Done — 2/2 cases evaluated

=== Answer Relevancy: 1.0000 (avg over 2 samples) ===
Completed: 200/200 rows


In [41]:
answer_relevancy_df['Answer Relevancy'].mean()

np.float64(0.9469989378181008)

In [ ]:
# Defining Answer Correctness
evaluation_steps = [
    "Compare the generated answer with the reference answer in the context of the original biomedical question.",
    "Check whether the generated answer contains factually correct biomedical information and no contradictions to the reference answer.",
    "Verify that all clinically important facts needed to answer the question are present and no critical information is missing.",
    "Ignore wording differences, but penalize incorrect medical claims, unsupported conclusions, or misleading clinical interpretations."
]
deepeval_ac = evaluate_deepeval_parallel(
    test_cases, GEval, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"vanilla_llm_critic_answer_correctness_{timestamp}.csv"),
    metric_kwargs={
        "name": "AnswerCorrectness",
        "evaluation_steps": evaluation_steps,
        "evaluation_params": [
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT,
            LLMTestCaseParams.EXPECTED_OUTPUT,
        ]}
)


200 cases split across 10 key(s) (20 cases/key max):
  Key 0: cases 0–19 (20 cases)
  Key 1: cases 20–39 (20 cases)
  Key 2: cases 40–59 (20 cases)
  Key 3: cases 60–79 (20 cases)
  Key 4: cases 80–99 (20 cases)
  Key 5: cases 100–119 (20 cases)
  Key 6: cases 120–139 (20 cases)
  Key 7: cases 140–159 (20 cases)
  Key 8: cases 160–179 (20 cases)
  Key 9: cases 180–199 (20 cases)



[Key 8] Done — 20/20 cases evaluated

=== AnswerCorrectness [GEval]: 0.3210 (avg over 200 samples) ===
Saved to /content/results/deepeval/vanilla_llm_critic_answer_correctness_20260510_140037.csv
